# Análise Exploratória

In [1]:
import sys
import numpy as np
import seaborn as sns
import pandas as pd
np.set_printoptions(threshold=sys.maxsize)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 80)
def id_like_dtypes(path):
    """Detecta colunas de identificação/código, CPF e CNPJ pelo nome e monta
    um dict de dtype=str para evitar perda de zeros à esquerda na leitura."""
    header = pd.read_csv(path, nrows=0).columns
    dtypes = {}
    for col in header:
        c = col.lower()
        is_id = (
            "codigo" in c or "cnpj" in c or "cpf" in c
            or c.startswith(("id", "numero", "sequencial", "cod"))
            or c.endswith(("id", "idpncp"))
            or c in {"nifornecedor"}
        )
        if is_id:
            dtypes[col] = str
    return dtypes


def read_csv_ids_str(path, dtype=None, **kwargs):
    """pd.read_csv que força colunas de identificação/código/CPF/CNPJ como string."""
    dtypes = id_like_dtypes(path)
    if dtype:
        dtypes.update(dtype)
    return pd.read_csv(path, dtype=dtypes, **kwargs)

In [2]:
raw_grupo_material = read_csv_ids_str('raw_grupo_material_202607071234.csv')
raw_classe_material = read_csv_ids_str('raw_classe_material_202607071232.csv')
raw_pdm_material = read_csv_ids_str('raw_pdm_material_202607071236.csv')
raw_item_material = read_csv_ids_str('raw_item_material_202607071235.csv')
raw_caracteristicas_material = read_csv_ids_str('raw_caracteristicas_material_202607071232.csv', low_memory=False)
raw_unidade_fornecimento_material = read_csv_ids_str('raw_unidade_fornecimento_material_202607071236.csv')
raw_natureza_despesa_material = read_csv_ids_str('raw_natureza_despesa_material_202607071235.csv')

Documentação: 
- https://github.com/GovHub-br/data-application-mgi/blob/main/documentacao-schema-compras-gov.md
- https://www.gov.br/compras/pt-br/acesso-a-informacao/manuais/manual-dados-abertos/manual-api-compras.pdf

# Domínio 1: Catálogo de Materiais

Hierarquia: **Grupo → Classe → PDM (Padrão Descritivo de Material) → Item**











```mermaid
erDiagram
    raw_grupo_material {
        text codigogrupo PK
        text nomegrupo
        text statusgrupo
    }
    raw_classe_material {
        text codigoclasse PK
        text codigogrupo FK
        text nomeclasse
        text statusclasse
    }
    raw_pdm_material {
        text codigopdm PK
        text codigoclasse FK
        text nomepdm
        text statuspdm
    }
    raw_item_material {
        text codigoitem PK
        text codigopdm FK
        text descricaoitem
        text statusitem
        text codigo_ncm
    }
    raw_caracteristicas_material {
        text codigoitem FK
        text codigocaracteristica
        text nomecaracteristica
        text codigovalorcaracteristica
        text nomevalorcaracteristica
    }
    raw_unidade_fornecimento_material {
        text codigopdm FK
        text siglaunidadefornecimento
        text nomeunidadefornecimento
    }
    raw_natureza_despesa_material {
        text codigopdm FK
        text codigonaturezadespesa
        text nomenaturezadespesa
    }

    raw_grupo_material ||--o{ raw_classe_material : "codigogrupo"
    raw_classe_material ||--o{ raw_pdm_material : "codigoclasse"
    raw_pdm_material ||--o{ raw_item_material : "codigopdm"
    raw_item_material ||--o{ raw_caracteristicas_material : "codigoitem"
    raw_pdm_material ||--o{ raw_unidade_fornecimento_material : "codigopdm"
    raw_pdm_material ||--o{ raw_natureza_despesa_material : "codigopdm"
```


## `raw_grupo_material`
Nível raiz da hierarquia de materiais.

| Coluna | Descrição |
|---|---|
| `codigogrupo` **PK** | Código identificador do grupo |
| `nomegrupo` | Nome do grupo |
| `statusgrupo` | Ativo/inativo |
| `datahoraatualizacao` | Última atualização na API |

In [3]:
raw_grupo_material.shape

(79, 5)

### `codigogrupo`
Código do grupo do material

In [4]:
raw_grupo_material.codigogrupo.duplicated().sum()

0

#### 79 grupos

In [5]:
raw_grupo_material.duplicated().sum()

0

In [6]:
raw_grupo_material.head()

,codigogrupo,nomegrupo,statusgrupo,datahoraatualizacao,dt_ingest
0,32,MÁQUINAS E EQUIPAMENTOS PARA TRABALHOS EM MADEIRA,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497
1,34,MAQUINAS PARA TRABALHO EM METAIS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497
2,35,EQUIPAMENTOS COMERCIAIS E DE SERVIÇOS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497
3,36,MÁQUINAS PARA INDÚSTRIAS ESPECIALIZADAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497
4,37,MÁQUINAS E EQUIPAMENTOS AGRÍCOLAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497


### `statusgrupo`
Status do grupo (0 – False/Inativo, 1 – True/Ativo)

In [7]:
raw_grupo_material.statusgrupo.value_counts(dropna=False)

statusgrupo
True     78
False     1
Name: count, dtype: int64

In [8]:
raw_grupo_material[raw_grupo_material['statusgrupo'] == False]

,codigogrupo,nomegrupo,statusgrupo,datahoraatualizacao,dt_ingest
63,64,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",False,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497


#### Apenas um grupo Inativo

## `raw_classe_material`
Subdivisão de grupos.

| Coluna | Descrição |
|---|---|
| `codigoclasse` **PK** | Código identificador da classe |
| `codigogrupo` **FK** → `raw_grupo_material` | Grupo pai |
| `nomeclasse` | Nome da classe |
| `statusclasse` | Ativo/inativo |

In [9]:
raw_classe_material.shape

(711, 7)

### `codigoclasse`
Código da classe do material

In [10]:
raw_classe_material.codigoclasse.duplicated().sum()

0

#### 711 classes

In [11]:
raw_classe_material.duplicated().sum()

0

In [12]:
raw_classe_material.head()

,codigoclasse,codigogrupo,nomegrupo,nomeclasse,statusclasse,datahoraatualizacao,dt_ingest
0,1015,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATÉ 125MM,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502
1,1020,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATÉ 150MM,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502
2,1025,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM ATÉ 200MM,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502
3,1030,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 200MM ATÉ 300MM,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502
4,1035,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 300MM,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502


### `statusclasse`
Status da Classe (0 – False/Inativo, 1 – True/Ativo)

In [13]:
raw_classe_material.statusclasse.value_counts(dropna=False)

statusclasse
True     658
False     53
Name: count, dtype: int64

#### 53 classes inativas

In [14]:
materiais = pd.merge(raw_grupo_material, raw_classe_material, on='codigogrupo', how='outer',
                        indicator='_grupo_classe', suffixes=('_grupo', '_classe'))

In [15]:
materiais._grupo_classe.value_counts()

_grupo_classe
both          711
left_only       0
right_only      0
Name: count, dtype: int64

In [16]:
materiais.shape

(711, 12)

## `raw_pdm_material`
Padrão Descritivo de Material — agrupamento de itens com características similares.

| Coluna | Descrição |
|---|---|
| `codigopdm` **PK** | Código do PDM |
| `codigoclasse` **FK** → `raw_classe_material` | Classe pai |
| `nomepdm` | Nome do PDM |
| `codigogrupo` | Grupo pai |
| `nomegrupo` | Nome do grupo |
| `nomeclasse` | Nome da classe |
| `statuspdm` | Ativo/inativo |


In [17]:
raw_pdm_material.shape

(20415, 9)

### `codigopdm`
Código do PDM

In [18]:
raw_pdm_material.codigopdm.duplicated().sum()

0

#### 20k PDMs

In [19]:
raw_pdm_material.duplicated().sum()

0

In [20]:
raw_pdm_material.head()

,codigogrupo,nomegrupo,codigoclasse,nomeclasse,codigopdm,nomepdm,statuspdm,datahoraatualizacao,dt_ingest
0,70,"INFORMÁTICA - EQUIPAMENTOS, PEÇAS, ACESSÓRIOS E SUPRIMENTOSDE TIC",7030,EQUIPAMENTOS DE ARMAZENAMENTO DE DADOS,242,TORRE CD ROM,True,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914
1,74,"MÁQUINAS PARA ESCRITÓRIO, SISTEMAS DE PROCESSAMENTO DE TEX-TO E FICHÁRIOS D...",7430,MÁQUINAS DE ESCREVER E DE COMPOSIÇÃO PARA ESCRITÓRIO,301,MÁQUINA ESCREVER,True,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914
2,74,"MÁQUINAS PARA ESCRITÓRIO, SISTEMAS DE PROCESSAMENTO DE TEX-TO E FICHÁRIOS D...",7490,MÁQUINAS DIVERSAS PARA ESCRITÓRIO,302,MÁQUINA PROTOCOLADORA,True,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914
3,74,"MÁQUINAS PARA ESCRITÓRIO, SISTEMAS DE PROCESSAMENTO DE TEX-TO E FICHÁRIOS D...",7490,MÁQUINAS DIVERSAS PARA ESCRITÓRIO,294,IMPRESSORA CHEQUE,True,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914
4,74,"MÁQUINAS PARA ESCRITÓRIO, SISTEMAS DE PROCESSAMENTO DE TEX-TO E FICHÁRIOS D...",7435,EQUIPAMENTOS DE SISTEMAS DE INFORMAÇÕES PARA ESCRITÓRIO,296,LEITORA MICROFICHAS,True,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914


### `statuspdm`
Status do PDM (0 – False/Inativo, 1 – True/Ativo)

In [21]:
raw_pdm_material.statuspdm.value_counts(dropna=False)

statuspdm
True     15018
False     5397
Name: count, dtype: int64

#### Existem 15k de PDMs ativos e 5,4k inativos

In [22]:
materiais = pd.merge(materiais, raw_pdm_material, on='codigoclasse', how='outer', indicator='_pdm', suffixes=('', '_pdm'))

In [23]:
materiais.shape

(20431, 21)

In [24]:
materiais._pdm.value_counts()

_pdm
both          20415
left_only        16
right_only        0
Name: count, dtype: int64

In [25]:
materiais[materiais['_pdm'] == 'left_only']

,codigogrupo,nomegrupo_grupo,statusgrupo,datahoraatualizacao_grupo,dt_ingest_grupo,codigoclasse,nomegrupo_classe,nomeclasse,statusclasse,datahoraatualizacao_classe,dt_ingest_classe,_grupo_classe,codigogrupo_pdm,nomegrupo,nomeclasse_pdm,codigopdm,nomepdm,statuspdm,datahoraatualizacao,dt_ingest,_pdm
210,13,MUNIÇÕES E EXPLOSIVOS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,1365,MUNIÇÕES E EXPLOSIVOS,PRODUTOS QUIMICOS DE USO MILITAR,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
656,23,VEíCULOS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,2315,VEíCULOS,VEICULOS DE SERVICOS ESPECIAIS,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2037,34,MAQUINAS PARA TRABALHO EM METAIS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,3440,MAQUINAS PARA TRABALHO EM METAIS,MATERIAIS E EQUIPAMENTOS PARA FUNDICAO,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2869,37,MÁQUINAS E EQUIPAMENTOS AGRÍCOLAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,3760,MÁQUINAS E EQUIPAMENTOS AGRÍCOLAS,VEICULOS DE TRACAO ANIMAL E REBOQUES PARA USO AGRICOLA,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6788,60,"MATERIAIS, COMPONENTES, CONJUNTOS E ACESSÓRIOS DE FIBRAS Ó-TICAS",True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,6034,"MATERIAIS, COMPONENTES, CONJUNTOS E ACESSÓRIOS DE FIBRAS Ó-TICAS",MODULADORES E DEMODULADORES DE FIBRA OTICA,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6790,60,"MATERIAIS, COMPONENTES, CONJUNTOS E ACESSÓRIOS DE FIBRAS Ó-TICAS",True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,6040,"MATERIAIS, COMPONENTES, CONJUNTOS E ACESSÓRIOS DE FIBRAS Ó-TICAS",SENSORES DE FIBRA OTICA,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7555,64,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",False,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,6440,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",MEDICAMENTOS PARA IMUNOTERAPIA E ALERGIA,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7559,64,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",False,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,6460,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",MEDICAMENTOS PARA AGENTES DIAGNOSTICOS,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7562,64,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",False,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,6475,"MEDICAMENTOS, DROGAS E PRODUTOS BIOLOGICOS",PRODUTOS QUIMICOS,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
12300,65,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁRIO E VETERINÁRIO",True,2026-06-10T11:28:28.238096,2026-06-26T16:35:45.646497,6555,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁRIO E VETERINÁRIO",JOGOS E CONJUNTOS MEDICOS,False,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


#### 16 Classes não tem PDM

## `raw_item_material`
Itens individuais do catálogo de materiais (343 mil itens).

| Coluna | Descrição |
|---|---|
| `codigoitem` **PK** | Código do item no catálogo |
| `codigopdm` **FK** → `raw_pdm_material` | PDM pai |
| `descricaoitem` | Descrição completa do item |
| `statusitem` | Ativo/inativo |
| `itemsustentavel` | Indicador de sustentabilidade |
| `codigo_ncm` | Código NCM (Nomenclatura Comum do Mercosul) |
| `descricao_ncm` | Descrição da classificação NCM |
| `aplica_margem_preferencia` | Aplica margem de preferência nacional |


In [26]:
raw_item_material.shape

(343036, 15)

### `codigoitem`
Código do item

In [27]:
raw_item_material.codigoitem.duplicated().sum()

0

#### 343k itens

In [28]:
raw_item_material.duplicated().sum()

0

In [29]:
raw_item_material.head()

,codigoitem,codigogrupo,nomegrupo,codigoclasse,nomeclasse,codigopdm,nomepdm,descricaoitem,statusitem,itemsustentavel,codigo_ncm,descricao_ncm,aplica_margem_preferencia,datahoraatualizacao,dt_ingest
0,206504,71,MOBILIÁRIOS,7110,MOBILIÁRIO PARA ESCRITÓRIO,313,CADEIRA ESCRITÓRIO,"CADEIRA ESCRITÓRIO, MATERIAL ESTRUTURA: TUBO AÇO , MATERIAL REVESTIMENTO ASS...",True,False,NaN,NaN,NaN,2021-10-16T09:43:08.030221,2026-06-26T16:54:56.207446
1,243756,71,MOBILIÁRIOS,7110,MOBILIÁRIO PARA ESCRITÓRIO,313,CADEIRA ESCRITÓRIO,"CADEIRA ESCRITÓRIO, MATERIAL REVESTIMENTO ASSENTO E ENCOSTO: TECIDO 100% LÃ ...",True,False,NaN,NaN,NaN,2021-10-16T09:43:08.030221,2026-06-26T16:54:56.207446
2,399016,68,SUBSTÂNCIAS E PRODUTOS QUÍMICOS,6810,PRODUTOS QUÍMICOS,10081,BETA-NICOTINAMIDA ADENINA DINUCLEOTÍDEO,"BETA-NICOTINAMIDA ADENINA DINUCLEOTÍDEO, ASPECTO FÍSICO: PÓ BRANCO HIGROSCÓP...",True,False,NaN,NaN,NaN,2021-10-16T09:43:08.030221,2026-06-26T16:54:56.207446
3,411745,65,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁRIO E VETERINÁRIO",6510,MATERIAIS CIRÚRGICOS PARA CURATIVOS,439,FITA HIPOALERGÊNICA,"FITA HIPOALERGÊNICA, TIPO: DORSO EM TECIDO A BASE DE RAYON ACETATO , LARGURA...",False,False,NaN,NaN,NaN,2021-10-16T09:43:08.030221,2026-06-26T16:54:56.207446
4,303481,65,"EQUIPAMENTOS E ARTIGOS PARA USO MÉDICO, DENTÁRIO E VETERINÁRIO",6510,MATERIAIS CIRÚRGICOS PARA CURATIVOS,468,CAMPO OPERATÓRIO,"CAMPO OPERATÓRIO, TIPO: DUPLO , COMPRIMENTO: 1,60 M, LARGURA: 110 CM, MATERI...",False,False,NaN,NaN,NaN,2021-10-16T09:43:08.030221,2026-06-26T16:54:56.207446


### `statusitem`
Status do item (0 – False/Inativo, 1 – True/Ativo)

In [30]:
raw_item_material.statusitem.value_counts(dropna=False)

statusitem
True     247552
False     95484
Name: count, dtype: int64

#### 95k itens inativos e 247k ativos

### `itemsustentavel`
Indica se o item é sustentável (0 – False/não, 1 – True/sim)

In [31]:
raw_item_material.itemsustentavel.value_counts(dropna=False)

itemsustentavel
False    342453
True        583
Name: count, dtype: int64

#### Apenas 583 itens sustentáveis

### `codigo_ncm`
Código de NCM – Nomenclatura Comum do Mercosul

In [32]:
raw_item_material.codigo_ncm.notna().sum()

44307

In [33]:
raw_item_material.codigo_ncm.value_counts(dropna=False)

codigo_ncm
NaN         298729
90189099      5283
90183929      1851
84439933      1777
30049099      1312
84439923      1270
90211020      1214
90183999      1063
30049069       991
30061090       888
85362000       777
85365090       771
85366990       759
30051090       672
84439199       613
90184919       503
8473           461
30049079       458
90189010       451
90213190       436
90184999       428
94029090       421
30059090       394
90219080       388
85361000       381
30049039       361
85366910       347
8517           337
90183219       325
90213920       322
90211010       314
85381000       301
90185090       277
85469000       264
73089010       261
85323090       236
90181910       235
30045090       235
90211099       234
85423190       232
61151099       226
85364          224
90184920       222
90200090       218
9032           215
85444200       215
90191000       212
84717010       207
38089199       207
30069110       205
8507           192
8423           189
8

#### 44k itens com NCM (inclusive repetidos)

### `aplica_margem_preferencia`
Indica se o item aplica margem de preferência (0 – False/Não, 1 – True/Sim)

In [34]:
raw_item_material.aplica_margem_preferencia.value_counts(dropna=False)

aplica_margem_preferencia
NaN      294152
False     33419
True      15465
Name: count, dtype: int64

#### 15k+ de itens em que se aplica a margem de preferência, contra 33k que não.

## Domínio do Catálogo de Materiais

In [35]:
materiais = pd.merge(materiais, raw_item_material, on='codigopdm', how='outer', indicator='_item', suffixes=('', '_item'))

In [36]:
materiais._item.value_counts()

_item
both          343036
left_only         99
right_only         0
Name: count, dtype: int64

In [37]:
materiais[materiais['_item'] == 'left_only'].head()

,codigogrupo,nomegrupo_grupo,statusgrupo,datahoraatualizacao_grupo,dt_ingest_grupo,codigoclasse,nomegrupo_classe,nomeclasse,statusclasse,datahoraatualizacao_classe,dt_ingest_classe,_grupo_classe,codigogrupo_pdm,nomegrupo,nomeclasse_pdm,codigopdm,nomepdm,statuspdm,datahoraatualizacao,dt_ingest,_pdm,codigoitem,codigogrupo_item,nomegrupo_item,codigoclasse_item,nomeclasse_item,nomepdm_item,descricaoitem,statusitem,itemsustentavel,codigo_ncm,descricao_ncm,aplica_margem_preferencia,datahoraatualizacao_item,dt_ingest_item,_item
2271,87,SUPRIMENTOS AGRÍCOLAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,8710,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,87,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,1014,RAÇÃO BALANCEADA FRANGO,False,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2524,87,SUPRIMENTOS AGRÍCOLAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,8710,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,87,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,1019,RAÇÃO BALANCEADA GALINHA,False,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2750,87,SUPRIMENTOS AGRÍCOLAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,8710,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,87,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,1020,RAÇÃO BALANCEADA PINTO,False,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2780,87,SUPRIMENTOS AGRÍCOLAS,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,8710,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,87,SUPRIMENTOS AGRÍCOLAS,FORRAGENS E ALIMENTOS,1021,RAÇÃO BALANCEADA SUÍNA,False,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
60462,75,UTENSÍLIOS DE ESCRITÓRIO E MATERIAL DE EXPEDIENTE,True,2021-10-16T09:16:33.723625,2026-06-26T16:35:45.646497,7510,UTENSÍLIOS DE ESCRITÓRIO E MATERIAL DE EXPEDIENTE,ARTIGOS PARA ESCRITÓRIO,True,2021-10-16T09:17:13.045775,2026-06-26T16:35:47.701502,both,75,UTENSÍLIOS DE ESCRITÓRIO E MATERIAL DE EXPEDIENTE,ARTIGOS PARA ESCRITÓRIO,135,GRAMPO TRILHO ENCADERNADORES,False,2021-10-16T09:21:41.961529,2026-06-26T16:36:33.033914,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


## `raw_caracteristicas_material` (sem PK)
Características e valores possíveis para cada item (1,9M de linhas — maior tabela do schema).

| Coluna | Descrição |
|---|---|
| `codigoitem` **FK** → `raw_item_material` | Item ao qual a característica pertence |
| `codigocaracteristica` | Código da característica |
| `nomecaracteristica` | Ex: "Cor", "Material", "Capacidade" |
| `codigovalorcaracteristica` | Código do valor |
| `nomevalorcaracteristica` | Ex: "Azul", "Aço inox", "500ml" |
| `numerocaracteristica` | Número de ordem da característica no item |
| `siglaunidademedida` | Unidade de medida (quando aplicável) |
| `itemsustentavel` | Flag de item sustentável |

> **Nota:** Sem chave primária definida — a API retorna dados com campos-chave nulos.

In [38]:
raw_caracteristicas_material.shape

(1904256, 13)

In [39]:
raw_caracteristicas_material.codigoitem.value_counts(dropna=False)

codigoitem
NaN    1904256
Name: count, dtype: int64

In [40]:
raw_caracteristicas_material.head()

,codigoitem,itemsustentavel,statusitem,codigocaracteristica,nomecaracteristica,statuscaracteristica,codigovalorcaracteristica,nomevalorcaracteristica,statusvalorcaracteristica,numerocaracteristica,siglaunidademedida,datahoraatualizacao,dt_ingest
0,NaN,False,True,NaN,MATERIAL,NaN,A00476,PAPEL KRAFT,True,NaN,FL,2021-10-16T09:43:08.030221,2026-07-01T11:30:35.893598
1,NaN,False,True,NaN,GRAMATURA,NaN,A00430,110,True,NaN,G/M2,2021-10-16T09:43:08.030221,2026-07-01T11:30:35.893598
2,NaN,False,True,NaN,TIPO,NaN,A00482,RPC PADRONIZADO,True,NaN,NaN,2021-10-16T09:43:08.030221,2026-07-01T11:30:35.893598
3,NaN,False,True,NaN,COMPRIMENTO,NaN,A00551,280,True,NaN,MM,2021-10-16T09:43:08.030221,2026-07-01T11:30:35.893598
4,NaN,False,True,NaN,COR,NaN,A00488,PARDA,True,NaN,NaN,2021-10-16T09:43:08.030221,2026-07-01T11:30:35.893598


## `raw_unidade_fornecimento_material` (sem PK)
Unidades de fornecimento aceitas por PDM (ex: "Caixa c/ 12 unidades").

| Coluna | Descrição |
|---|---|
| `codigopdm` **FK** → `raw_pdm_material` | PDM ao qual a unidade pertence |
| `siglaunidadefornecimento` | Sigla (ex: CX, PC, KG) |
| `nomeunidadefornecimento` | Nome completo |
| `descricaounidadefornecimento` | Descrição detalhada |
| `siglaunidademedida` | Unidade de medida base |
| `capacidadeunidadefornecimento` | Quantidade por unidade de fornecimento |

> **Nota:** Sem chave primária definida — a API retorna dados com campos-chave nulos.|

In [41]:
raw_unidade_fornecimento_material.shape

(47990, 10)

In [42]:
raw_unidade_fornecimento_material.head()

,codigopdm,siglaunidadefornecimento,nomeunidadefornecimento,descricaounidadefornecimento,siglaunidademedida,capacidadeunidadefornecimento,numerosequencialunidadefornecimento,statusunidadefornecimentopdm,datahoraatualizacao,dt_ingest
0,NaN,NaN,UNIDADE,UMA QUANTIDADE NORMALIZADA OU NUMERICA DE UM ITEM DE ABASTECIMENTO. NÃO UTI...,NaN,NaN,NaN,True,2021-10-16T09:30:54.651407,2026-06-26T16:51:56.876759
1,NaN,NaN,UNIDADE,UMA QUANTIDADE NORMALIZADA OU NUMERICA DE UM ITEM DE ABASTECIMENTO. NÃO UTI...,NaN,NaN,NaN,True,2021-10-16T09:30:54.651407,2026-06-26T16:51:56.876759
2,NaN,NaN,UNIDADE,UMA QUANTIDADE NORMALIZADA OU NUMERICA DE UM ITEM DE ABASTECIMENTO. NÃO UTI...,NaN,NaN,NaN,True,2021-10-16T09:30:54.651407,2026-06-26T16:51:56.876759
3,NaN,NaN,UNIDADE,UMA QUANTIDADE NORMALIZADA OU NUMERICA DE UM ITEM DE ABASTECIMENTO. NÃO UTI...,NaN,NaN,NaN,True,2021-10-16T09:30:54.651407,2026-06-26T16:51:56.876759
4,NaN,NaN,UNIDADE,UMA QUANTIDADE NORMALIZADA OU NUMERICA DE UM ITEM DE ABASTECIMENTO. NÃO UTI...,NaN,NaN,NaN,True,2021-10-16T09:30:54.651407,2026-06-26T16:51:56.876759


## `raw_natureza_despesa_material` (sem PK)
Naturezas de despesa orçamentária associadas a PDMs.

| Coluna | Descrição |
|---|---|
| `codigopdm` **FK** → `raw_pdm_material` | PDM ao qual a natureza pertence |
| `codigonaturezadespesa` | Código da natureza de despesa (ex: 33903000) |
| `nomenaturezadespesa` | Nome da natureza de despesa |

> **Nota:** Sem chave primária definida — a API retorna dados com campos-chave nulos.

In [43]:
raw_natureza_despesa_material.shape

(1574, 5)

In [44]:
raw_natureza_despesa_material.duplicated().sum()

1476

In [45]:
raw_natureza_despesa_material.notnull().sum()

codigopdm                   0
codigonaturezadespesa       0
nomenaturezadespesa      1561
statusnaturezadespesa    1561
dt_ingest                1574
dtype: int64

In [46]:
raw_natureza_despesa_material.drop_duplicates(inplace=True)

In [47]:
raw_natureza_despesa_material.head()

,codigopdm,codigonaturezadespesa,nomenaturezadespesa,statusnaturezadespesa,dt_ingest
0,NaN,NaN,NaN,NaN,2026-06-26T16:46:48.916449
13,NaN,NaN,"MATERIAL DE CONSUMO - UNIFORMES, TECIDOS E AVIAMENTOS",True,2026-06-26T16:46:48.916449
16,NaN,NaN,MATERIAL DE CONSUMO - MATERIAL DE TIC (CONSUMO),True,2026-06-26T16:46:48.916449
19,NaN,NaN,MATERIAL DE CONSUMO - MATERIAL HOSPITALAR,True,2026-06-26T16:46:48.916449
42,NaN,NaN,MATERIAL DE CONSUMO - MATERIAL BIBLIOGRAFICO,True,2026-06-26T16:46:48.916449
